# Lesson 12 - Pagpapaliit ng Kasaysayan ng Chat gamit ang Agent Scratchpad

Ipinapakita sa notebook na ito kung paano pamahalaan ang konteksto sa mahahabang pag-uusap gamit ang Microsoft Agent Framework. Habang lumalaki ang pag-uusap, tumataas ang bilang ng mga token — na kalaunan ay lalampas sa window ng konteksto ng modelo. Nilulutas namin ito gamit ang isang **pattern ng pagbubuod ng konteksto** at isang **agent scratchpad** para sa permanenteng memorya.

## Mga Matututunan Mo:
1. **Bakit Mahalaga ang Pamamahala ng Konteksto**: Pag-unawa sa mga limitasyon ng token at mga window ng konteksto
2. **Mga Context-Aware Agents**: Paggawa ng mga agent na namamahala ng kanilang sariling konteksto ng pag-uusap
3. **Pattern ng Pagbubuod ng Konteksto**: Paggamit ng mga kasangkapan upang paikliin ang kasaysayan ng pag-uusap
4. **Agent Scratchpad**: Permanenteng memorya na nananatili sa kabila ng pagpapaliit ng konteksto

## Mga Kinakailangan:
- Azure OpenAI na naka-setup kasama ang mga naka-configure na environment variables
- Pag-unawa sa mga pangunahing konsepto ng agent mula sa mga naunang aralin


## Pag-setup


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Bakit Mahalaga ang Pamamahala ng Konteksto

Bawat LLM ay may hangganang **context window** — ang pinakamataas na bilang ng mga token na kaya nitong iproseso sa isang kahilingan. Habang umuusad ang isang multi-turn na pag-uusap:

- **Lumalaki nang linya ang bilang ng token** sa bawat mensahe ng user at sagot ng assistant.
- **Nangunguna sa gastos ang mga prompt token** dahil ang buong kasaysayan ay muling ipinapadala sa bawat turn.
- Sa kalaunan ang pag-uusap ay **lalampas sa context window** at ang modelo ay magpuputol o magbibigay ng error.

### Mga Estratehiya sa Pamamahala ng Konteksto

| Estratehiya | Paano Ito Gumagana | Kapalit |
|---|---|---|
| **Pagpuputol** | Tanggalin ang pinakamatandang mga mensahe | Nawawala ang unang konteksto |
| **Pagsusuma** | Pagsiksikin ang mas lumang mga mensahe sa isang buod | May nawalang detalye, pero napanatili ang mga pangunahing punto |
| **Scratchpad / Panlabas na Memorya** | Itago ang mga susi na impormasyon sa labas ng pag-uusap | Nangangailangan ng tool calls, pero nananatili kahit bumaba ang datos |

Sa notebook na ito pinaghalo namin ang **pagsusuma** sa isang **scratchpad tool** upang mapanatili ng agent ang tuloy-tuloy kahit na paikliin ang kasaysayan ng pag-uusap.


## Paglikha ng Isang Ahenteng Marunong sa Konteksto


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Pagsasagawa ng Mahabang Pag-uusap

Dumaan tayo sa isang pag-uusap na may maraming palitan upang makita kung paano nag-iipon ang konteksto. Dapat panatilihin ng ahente ang mahahalagang detalye (mga kagustuhan, badyet, mga petsa ng paglalakbay) sa bawat palitan at ipakita ang tuloy-tuloy na daloy.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Pansinin kung paano naaalala ng ahente ang konteksto mula sa mga naunang usapan — alam nito ang tungkol sa Japan, sushi, mga templo, potograpiya, ang $3000 na badyet, paglalakbay nang mag-isa, at ang buwan ng Abril bilang panahon. Sa isang maikling usapan, maganda ang takbo nito, pero habang lumalaki ang usapan, nagiging mahal ang pagpapadala muli ng buong kasaysayan.

Ipagpatuloy natin ang usapan nang may dagdag na mga usapan upang makita ang pag-ipon ng konteksto:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Pattern ng Pagbubuod ng Konteksto

Habang lumalago ang pag-uusap, maaari nating gamitin ang isang **tool sa pagbubuod** upang paikliin ang naipon na konteksto sa isang compact na anyo. Tinatawag ng ahente ang tool na ito upang itala ang mga pangunahing kagustuhan upang kahit na tanggalin ang mga lumang mensahe, mapanatili ang mahahalagang impormasyon.

Ang pattern na ito ang pundasyon para sa mas sopistikadong pagbabawas ng kasaysayan:
1. Tinutukoy ng ahente ang mga pangunahing katotohanan mula sa pag-uusap
2. Tinatawag nito ang tool sa pagbubuod upang itala ang mga ito
3. Maaaring ligtas na alisin ang mga lumang mensahe dahil nasasaklaw ng pagbubuod ang mahahalaga

Sa ibaba ay tinutukoy namin ang isang `summarize_preferences` tool na maaaring tawagin ng ahente upang itala ang compact na buod ng mga natutunan nito.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Buod

Sa araling ito natutunan mo kung paano pamahalaan ang konteksto sa mahahabang pag-uusap ng ahente gamit ang Microsoft Agent Framework:

### Pangunahing Konsepto
- **May hangganan ang mga context window** — bawat token sa kasaysayan ng pag-uusap ay may bayad at binibilang sa limitasyon.
- **Pinapayagan ng mga kasangkapan sa pagsasummarize** ang ahente na paikliin ang naipon na konteksto sa mga compact na buod, na nagpapababa ng paggamit ng token habang pinananatili ang mahalagang impormasyon.
- **Ang mga scratchpad ng ahente** ay nagbibigay ng matatag na panlabas na memorya na nananatili kahit pa mabawasan ang pag-uusap.

### Ang Iyong Naitayo
- Isang **ahente na may kamalayan sa konteksto** na nagpapanatili ng tuloy-tuloy na daloy sa mga multi-turn na pag-uusap
- Isang **kasangkapang pang-summarize** (`summarize_preferences`) na nagtala ng mga pangunahing detalye ng gumagamit sa isang compact na format
- Isang **multi-turn na pag-uusap** na nagpapakita ng pagpapanatili ng konteksto at paghawak ng mga pagbabago

### Mga Aplikasyong Totoo sa Mundo
- **Mga Bot sa Serbisyo sa Customer**: Naaalala ang mga kagustuhan sa mahahabang sesyon ng suporta
- **Mga Personal na Katulong**: Natutunton ang mga patuloy na proyekto nang hindi na kailangang ipaliwanag muli ang konteksto
- **Mga Guro**: Pinapanatili ang progreso ng estudyante sa maraming pakikipag-ugnayan

### Mga Susunod na Hakbang
- Magpatupad ng kompletong tool na scratchpad na may pagsuporta sa file-based persistence
- Magdagdag ng awtomatikong pagtanggal ng kasaysayan pagkatapos mag-summarize
- Pagsamahin sa mga vector database para sa semantic memory search
- Gumawa ng mga ahente na maaaring ipagpatuloy ang pag-uusap makalipas ang ilang araw na may kompletong konteksto


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Paunawa**:
Ang dokumentong ito ay isinalin gamit ang AI translation service na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagamat nagsusumikap kami ng katumpakan, mangyaring tandaan na ang mga awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o di-tumpak na impormasyon. Ang orihinal na dokumento sa sariling wika nito ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang hindi pagkakaunawaan o maling interpretasyon na maaaring lumabas mula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
